# 🧪 PyTorch Lab 8: Distillation

> **Durée cible** : 1h30 
---

## 🎯 Objectifs
À la fin de ce lab, vous saurez :
1. Implémentez la distillation
2. Evaluer sa pertinence 

# Installation & Setup
* Import modules
* Device configuration

In [ ]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import base64

from torch import device, cuda, optim, max
from torchvision import transforms, datasets
from torchvision.transforms import ToTensor, transforms
from torch.utils.data import ConcatDataset, Subset, DataLoader, Dataset, random_split
import torch.nn.functional as F
import torch.nn as nn
import torch
from tqdm import tqdm, trange

In [ ]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Data Acquisition and Preprocessing
* Dataset download
* Normalize data
* Create dataloaders

In [ ]:
import os

os.environ['HTTP_PROXY'] = 'http://cache.univ-st-etienne.fr:3128/'
os.environ['HTTPS_PROXY'] = 'http://cache.univ-st-etienne.fr:3128/'

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize( (0.1307,), (0.3081,))])

train_data = datasets.MNIST(
    root = 'data',
    train = True,
    transform = transform,
    download = True
)

test_data = datasets.MNIST(
    root = 'data',
    train = False,
    transform = transform,
    download = True
)

In [ ]:
loaders = {
    'train' : DataLoader(train_data, batch_size=64, num_workers=16),
    'test'  : DataLoader(test_data, batch_size=64, num_workers=16)
}

# Teacher

We provide here the architecture of the Teacher model that we used and trained. 



In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return x

    def return_param(self):
        w = []
        for param in self.parameters():
            w.append(param.flatten())
        return torch.cat(w)

    def write_param(self,w):
        with torch.no_grad():
            for param in self.parameters():
                w_temp = w[:np.prod(np.asarray(param.shape))].squeeze()
                w_temp = torch.reshape(torch.Tensor(w_temp),param.shape)
                param.copy_(w_temp)
                w = w[np.prod(np.asarray(param.shape)):]

In [ ]:
with open("Data/data_30_0.p", "rb" ) as file:
    byte = file.read()
    decoded = base64.decodebytes(byte)
    w_data, wy_data, wacc_data = pickle.loads(decoded)

teacher = Net()
teacher.to(device)

teacher.write_param(w_data[29])

In [ ]:
def test_model(model):
    total = 10000
    correct = 0
    model.eval()
    for inputs, labels in tqdm(loaders['test'], desc = f'Test'):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [ ]:
print(f"\nTeacher accuracy on test dataset: {test_model(teacher)}%")

# Student 1

Now it's your turn to play :-) 

Implement a classic MLP with 2 layers train with soft labels, i.e. using the teacher output as target

# Student 2
Classic MLP with 2 layers train with hard labels, i.e. use the real labels oin one hot ratehr than the teacher output


# Fighing time 🔫

Compare the results, vizualise, analyse

# Accuracy by parameter
The teacher model have 21840 parameters.  
We will build and evaluate different MLP model structures to see how the number of parameters influences the accuracy of our models.  
Evaluate the evolution of accuracy wrt number of parameters : 7840, 9k, 10k, 11k, 12k, 13k, 14k, 15k, 16k, 17k, 18k, 19k, 20k, 21k

Our MLP has two layers: the input layer and the output layer, with fixed sizes of **784** and **10** respectively.  

Therefore, we can only adjust one value: the size of the hidden layer \( x \).

The number of parameters is computed as follows:

$
\text{nb\_param} = 784 \cdot x + x \cdot 10 
\quad \text{(with } x \text{ an integer)}
$

$
\Rightarrow \text{nb\_param} = x \cdot (784 + 10) = 794 \cdot x
$

$
\Rightarrow x = \frac{\text{nb\_param}}{794}
$

For example, if $nb\_param = 7840$:

$
x = \frac{7840}{794} = 9.87 \approx 10
$

In [16]:
nb_param = [7840, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000, 21000]

In [17]:
x = [int(i/794)+1 for i in nb_param]